# PFE Spark — Entraînement sklearn dans Snowflake
Entraîne un `LogisticRegression` pour prédire la **résolution** des tickets Apache Spark.
Lit directement depuis `MARTS_ML.MART_ML` (42 083 tickets, features tabulaires + `has_parent`).
Sauvegarde le modèle dans le **Snowflake Model Registry**.

In [ ]:
import pandas as pd
import numpy as np
from snowflake.snowpark.context import get_active_session
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, classification_report
import json

session = get_active_session()
print('Session Snowflake active :', session.get_current_database())

## 1. Chargement de MART_ML

In [ ]:
TABULAR_FEATURES = [
    'n_total_changes', 'n_status_changes', 'n_priority_changes',
    'n_assignee_changes', 'n_resolution_changes', 'was_escalated',
    'n_people_involved', 'n_links_total', 'n_duplicates', 'n_blocks',
    'n_blocked_by', 'n_relates', 'n_comments', 'n_commenters',
    'resolution_days', 'summary_length', 'description_length',
    'n_container', 'has_parent',
]
feat_str = ', '.join(TABULAR_FEATURES)

df = session.sql(f"""
    SELECT key, split, issuetype, resolution, {feat_str}
    FROM PFE_SPARK.MARTS_ML.MART_ML
    WHERE split IN ('train', 'validation')
    ORDER BY key
""").to_pandas()
df.columns = [c.lower() for c in df.columns]
print(f'Chargé {len(df):,} tickets')
print(df[['split','issuetype','resolution']].value_counts('split'))

## 2. Préparation des features

In [ ]:
RARE = {'Task', 'Documentation', 'Test', 'Question'}
df['issuetype'] = df['issuetype'].apply(lambda x: 'Other' if x in RARE else x)

train = df[df['split']=='train'].reset_index(drop=True)
val   = df[df['split']=='validation'].reset_index(drop=True)

scaler    = StandardScaler()
train_tab = scaler.fit_transform(train[TABULAR_FEATURES].fillna(0))
val_tab   = scaler.transform(val[TABULAR_FEATURES].fillna(0))

print(f'Train: {len(train):,}  Val: {len(val):,}')
print('Distribution resolution (train):')
print(train['resolution'].value_counts())

## 3. Entraînement LogisticRegression (résolution)

In [ ]:
clf_res = LogisticRegression(
    class_weight='balanced', max_iter=1000, C=3.0, n_jobs=-1
)
clf_res.fit(train_tab, train['resolution'])

pred = clf_res.predict(val_tab)
f1   = f1_score(val['resolution'], pred, average='macro', zero_division=0)
acc  = (pred == val['resolution'].values).mean()
print(f'Resolution — macro-F1={f1:.4f}  accuracy={acc:.4f}')
print(classification_report(val['resolution'], pred, zero_division=0))

## 4. Sauvegarde dans le Model Registry

In [ ]:
from snowflake.ml.registry import Registry

reg = Registry(session=session)
reg.log_model(
    clf_res,
    model_name='resolution_classifier',
    version_name='v1_tabular_has_parent',
    conda_dependencies=['scikit-learn', 'pandas', 'numpy'],
    comment=f'LR balanced, tabular only + has_parent. macro-F1={f1:.4f}'
)
print('Modèle sauvegardé dans Model Registry !')
print('Accessible via : SELECT * FROM SNOWFLAKE.ML.MODELS;')